<a href="https://colab.research.google.com/github/taek20230541/maritimedatamining/blob/main/Colab_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12주차 문제1


In [5]:
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
from pathlib import Path

# --- [준비 단계] 실습용 데이터 파일 생성 (파일이 없을 경우 대비) ---
DATA_DIR = Path('/content/type3')
DATA_DIR.mkdir(parents=True, exist_ok=True)
file_path = DATA_DIR / 'retail_sales_reg1.csv'

# 샘플 데이터 생성 (광고비, 직원수, 이벤트 횟수에 따른 매출 데이터)
np.random.seed(42)
n_samples = 50
ad_cost = np.random.randint(10, 100, n_samples)
staff = np.random.randint(2, 10, n_samples)
event_cnt = np.random.randint(0, 5, n_samples)
# 제공해주신 회귀식과 유사한 패턴으로 매출 생성
sales = 57.8 + 2.8 * ad_cost + 4.3 * staff + 20.5 * event_cnt + np.random.normal(0, 5, n_samples)

df_sample = pd.DataFrame({
    'sales': sales,
    'ad_cost': ad_cost,
    'staff': staff,
    'event_cnt': event_cnt
})
df_sample.to_csv(file_path, index=False)
print(f"파일 저장 완료: {file_path}\n")

# --- [실습 시작] 다중회귀분석 01 ---

# 1. 데이터 불러오기
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head(), '\n')

# 2. 회귀모형 적합 (Ordinary Least Squares)
# 공식: 종속변수(sales) ~ 독립변수들(ad_cost + staff + event_cnt)
model = smf.ols('sales ~ ad_cost + staff + event_cnt', data=df).fit()

# 3. 결과 요약 출력
print('[회귀 요약 결과]')
print(model.summary())

# 4. 핵심 지표 추출
print('\n' + '='*50)
print(f"{'항목':<15} | {'값':<10}")
print('-'*50)
print(f"{'R-squared':<15} | {model.rsquared:.4f}")
print(f"{'Adj. R-squared':<15} | {model.rsquared_adj:.4f}")
print(f"{'F-statistic':<15} | {model.fvalue:.4f}")
print(f"{'Prob (F-stat)':<15} | {model.f_pvalue:.4e}")
print('='*50)

# 5. 회귀계수 및 유의성 확인
print('\n[변수별 상세 결과]')
results_df = pd.concat([model.params, model.pvalues], axis=1)
results_df.columns = ['Coefficient', 'P-value']
print(results_df)

# 6. 해석 결과 출력
beta_ad = model.params['ad_cost']
print(f'\n💡 해석: 광고비(ad_cost)가 10단위 증가할 때, 매출은 약 {beta_ad * 10:.2f}단위 증가할 것으로 예상됩니다.')

sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
print(f'✅ 통계적으로 유의한 변수 (p < 0.05): {sig_vars}')

파일 저장 완료: /content/type3/retail_sales_reg1.csv

[데이터 상위 5행]
        sales  ad_cost  staff  event_cnt
0  333.796415       61      7          4
1  218.495969       24      3          4
2  317.700914       81      3          1
3  349.205869       70      2          4
4  184.171404       30      3          1 

[회귀 요약 결과]
                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.997
Model:                            OLS   Adj. R-squared:                  0.997
Method:                 Least Squares   F-statistic:                     5366.
Date:                Sun, 24 May 2026   Prob (F-statistic):           1.58e-58
Time:                        06:30:06   Log-Likelihood:                -144.22
No. Observations:                  50   AIC:                             296.4
Df Residuals:                      46   BIC:                             304.1
Df Model:                           3            

# 12주차 문제 2

In [6]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error
from pathlib import Path

# --- [준비 단계] 실습용 데이터 파일 생성 (파일이 없을 경우 대비) ---
DATA_DIR = Path('/content/type3')
DATA_DIR.mkdir(parents=True, exist_ok=True)
file_path = DATA_DIR / 'weather_reg2.csv'

# 샘플 데이터 생성 (기온, 일사량, 풍속, 오존 농도)
np.random.seed(42)
n_samples = 100
solar = np.random.randint(50, 300, n_samples)
wind = np.random.uniform(1, 15, n_samples)
o3 = np.random.randint(10, 80, n_samples)

# 핵심 설명에 명시된 계수를 기반으로 기온(temperature) 생성
# temperature = 11.78 + 0.04*solar - 0.92*wind + 0.24*o3 + 오차
temperature = 11.7807 + (0.0439 * solar) - (0.9220 * wind) + (0.2437 * o3) + np.random.normal(0, 2, n_samples)

df_sample = pd.DataFrame({
    'temperature': temperature,
    'solar': solar,
    'wind': wind,
    'o3': o3
})
df_sample.to_csv(file_path, index=False)
print(f"✅ 파일 생성 완료: {file_path}\n")

# --- [실습 시작] 다중회귀분석 02 ---

# 1. 데이터 불러오기
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head(), '\n')

# 2. 회귀모형 적합
model = smf.ols('temperature ~ solar + wind + o3', data=df).fit()

# 3. 결과 요약 (전체 레포트)
print('[회귀 요약 결과]')
print(model.summary())

# 4. 핵심 지표 출력
print('\n' + '='*50)
print(f"{'지표명':<15} | {'결과값':<15}")
print('-'*50)
print(f"{'R-squared':<15} | {model.rsquared:.4f}")
print(f"{'Adj. R-squared':<15} | {model.rsquared_adj:.4f}")
print('='*50)

# 5. 특정 조건 예측 (solar=100, wind=5, o3=30)
new_df = pd.DataFrame({
    'solar': [100],
    'wind': [5],
    'o3': [30]
})

# 예측 및 신뢰구간 포함 요약표
pred_res = model.get_prediction(new_df)
pred_frame = pred_res.summary_frame(alpha=0.05)

print('\n[특정 조건 예측 결과]')
print(f"조건: solar=100, wind=5, o3=30")
print(f"예측 기온: {pred_frame['mean'].values[0]:.4f}")
print(f"95% 신뢰구간: [{pred_frame['mean_ci_lower'].values[0]:.4f} ~ {pred_frame['mean_ci_upper'].values[0]:.4f}]")

# 6. 모델 성능 평가 (MSE / RMSE)
y_true = df['temperature']
y_pred = model.predict(df)

mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)

print('\n[모델 오차 지표]')
print(f"MSE  (평균제곱오차): {mse:.4f}")
print(f"RMSE (평균제곱근오차): {rmse:.4f}")

✅ 파일 생성 완료: /content/type3/weather_reg2.csv

[데이터 상위 5행]
   temperature  solar       wind  o3
0    27.093375    152   6.532103  62
1    23.294588    229  13.973224  53
2    18.643048    142  11.181808  41
3    28.559954     64   5.571571  79
4    18.641922    156   8.986216  41 

[회귀 요약 결과]
                            OLS Regression Results                            
Dep. Variable:            temperature   R-squared:                       0.914
Model:                            OLS   Adj. R-squared:                  0.911
Method:                 Least Squares   F-statistic:                     338.7
Date:                Sun, 24 May 2026   Prob (F-statistic):           6.45e-51
Time:                        06:30:55   Log-Likelihood:                -216.35
No. Observations:                 100   AIC:                             440.7
Df Residuals:                      96   BIC:                             451.1
Df Model:                           3                                       